# Zenodo tutorial

Upload and download resources for the **personalizedVEP** manuscript using the [zenodo_client](https://pypi.org/project/zenodo_client/) package and the helpers in `src/zenodo.py`.

## Prerequisites

1. **Install zenodo_client** (requires Python ≥3.10) and **python-dotenv** (to load token from `.env`):
   ```bash
   pip install zenodo_client python-dotenv
   ```

2. **Create a Zenodo access token** and put it in a **`.env`** file in the repo root:
   - **Production** (zenodo.org): https://zenodo.org/account/settings/applications/tokens/new/  
     In `.env`: `ZENODO_API_TOKEN=your_token`
   - **Sandbox** (testing only): https://sandbox.zenodo.org/account/settings/applications/tokens/new/  
     In `.env`: `ZENODO_SANDBOX_API_TOKEN=your_sandbox_token`  
     Sandbox and production are separate: a production token will not work with `sandbox=True` (403).

In [1]:
%load_ext autoreload
%autoreload 2

import os

# Run from repo root so paths like results/data resolve correctly
if 'NOTEBOOK_INITIALIZED' not in globals():
    os.chdir(os.path.dirname(os.path.abspath('.')))
    NOTEBOOK_INITIALIZED = True

import src.zenodo as zenodo

## Upload (recommended): draft first, then files

Create the draft once, then upload files. If uploads fail, you can re-run the upload step against the same draft without creating a new one.

**Step 1 — Create draft:** Creates an unpublished deposition and stores its id under the key `personalizedVEP` (in `~/.config/zenodo.ini`). If a draft for this key already exists, it is reused.

**Step 2 — Upload files:** Uploads all files from **`results/data/`** to that draft. You can re-run this cell if uploads fail; already-uploaded files are skipped by default (same key and size).

**Step 3 — Publish (optional):** When you're ready, publish the draft to get a DOI.

Use `sandbox=False` for production (zenodo.org); use `sandbox=True` only with a token from sandbox.zenodo.org.

In [2]:
# Step 1: Create the draft (run once; re-run is safe — reuses same draft if it exists)
dep = zenodo.create_draft(key="personalizedVEP", title="personalizedVEP", sandbox=False)
print("Draft id:", dep["id"])
print("State:", dep.get("state"))
# Edit link (replace zenodo.org with sandbox.zenodo.org if sandbox=True):
print("Edit:", dep.get("links", {}).get("html"))

Draft id: 18452091
State: unsubmitted
Edit: https://zenodo.org/deposit/18452091


In [ ]:
# Step 2: Upload files from results/data to the draft (can re-run if uploads failed)
# Uses the draft stored under key "personalizedVEP" from Step 1.
res = zenodo.upload_to_draft(key="personalizedVEP", dir_path="results/data", sandbox=False)
print("State:", res.get("state"))
print("Files:", len(res.get("files", [])))

Skipping 17 file(s) already present in draft (same key and size).


Uploading to Zenodo:   0%|          | 0/3 [00:00<?, ?file/s]

In [ ]:
# Step 3 (optional): Publish the draft to get a DOI
# pub = zenodo.publish_draft(key="personalizedVEP", sandbox=False)
# print("DOI:", pub.get("doi"))

## Alternative: one-shot upload

`upload_results_data()` creates the draft, uploads files, and publishes in one call. Use the draft-first flow above if you want to retry uploads without creating a new record.

In [ ]:
# res = zenodo.upload_results_data(sandbox=False)  # create + upload + publish in one go
# res.json() if hasattr(res, 'json') else res

## Upload as a single zip

Use `upload_zip_to_draft()` to pack several files or folders into one zip and upload that single file. Pass a dict mapping **paths inside the zip** → **local file or directory**. Directories are added with all contents under the given zip path.

In [ ]:
# Build a zip and upload it to the draft (create_draft first)
# items: path inside zip -> local file or dir (~ expanded internally)
res = zenodo.upload_zip_to_draft(
    items={
        "vep/esm/": "~/projects/VEP_protein/results/data/vep_df_esm*.parquet",           # contents of results/data/ go under data/ in zip 
        "vep/spliceai": "~/projects/data/1000_Genomes_on_GRCh38/SpliceAI/spliceai_clinvar_merged.parquet",
        "vep/flashzoi": "~/projects/data/1000_Genomes_on_GRCh38/clinvar_utr_snv/vep_df.parquet",
        "haplosaurus/": "~/projects/data/haplosaurus",
        "surrogate/esm/": "~/projects/VEP_protein/results/data/wtvariants_to_vep_*.pkl",
        "surrogate/spliceai/": "/home/schilder/projects/VEP_DNA/tmp/data/surrogate_models/SpliceAI/ridge_model_*.pkl",

        "misc/": "~/projects/VEP_protein/results/data/freq_df.parquet",
        "misc/": "~/projects/VEP_protein/results/data/normality_results.parquet",
        "misc/": "~/projects/VEP_protein/results/data/*_n_variants_df.parquet",
        "misc/": "~/projects/VEP_protein/results/data/vep_ecdf.parquet",
        "misc/": "~/projects/VEP_protein/results/data/vep_ecdf_umap.tsv.gz", 
        
    },
    key="personalizedVEP",
    zip_filename="personalizedVEP_data.zip",
    sandbox=False,
)
# To only build the zip locally (no upload): zenodo.build_zip_from_items(items, "out.zip")

Zip contents (before build):
  vep/esm: 9 file(s), 2.1 GB
  vep/spliceai: 1 file(s), 670.9 MB
  vep/flashzoi: 1 file(s), 697.9 MB
  haplosaurus: 6 file(s), 7.1 GB
  surrogate/esm: 3 file(s), 15.0 MB
  surrogate/spliceai: 302 file(s), 4.2 GB
  misc: 1 file(s), 1.7 MB
  TOTAL: 323 file(s), 14.7 GB


Building zip:   0%|          | 0/323 [00:00<?, ?file/s]

Create a separate draft for the colabfold data, as it takes up 19GB and has >8k files.

In [ ]:
res_fold = zenodo.upload_zip_to_draft(
    items={ 
        # WARNING: colabfold alone takes up 19GB and has >8k files
        "colabfold/": "~/projects/data/colabfold/",
    },
     key="personalizedVEP",
    zip_filename="personalizedFolding_data.zip",
    sandbox=False,
)

## Download: list and fetch files from a record

Once the record is published, you can fetch the latest version by **record id** (or **conceptrecid**).

In [ ]:
# Replace with your published record id (or conceptrecid for latest version)
RECORD_ID = "12345678"

# Get latest record metadata
record = zenodo.get_latest_record(RECORD_ID, sandbox=False)
print("Title:", record.get("metadata", {}).get("title"))
print("Concept recid:", record.get("conceptrecid"))
print("DOI:", record.get("doi"))

In [ ]:
# List files in the record
files = zenodo.list_record_files(RECORD_ID, sandbox=False)
for f in files:
    print(f.get("key") or f.get("filename"), "—", f.get("size", 0), "bytes")

In [ ]:
# Download a single file (stored in pystow cache; optional copy to dest_dir)
# path = zenodo.download_file(RECORD_ID, "filename.parquet", dest_dir="results/downloaded")
# print(path)

# Download all files into a directory
# paths = zenodo.download_record_files(RECORD_ID, "results/downloaded", sandbox=False)
# print(paths)

## Custom metadata

To set description or creators when creating the record, pass them into `upload_results_data()`:

In [ ]:
# from zenodo_client import Creator
#
# res = zenodo.upload_results_data(
#     description="My custom description.",
#     creators=[
#         Creator(name="Smith, Jane", affiliation="My University", orcid="0000-0000-0000-0000"),
#     ],
#     sandbox=True,
# )